# 05 · Demand weather analysis

Este notebook fecha **Q1** e **Q2** do plano:

- **Q1:** como a demanda reage à chuva em março/2026
- **Q2:** como essa resposta varia entre perfis de passageiro

**Janela usada aqui:** a janela completa de **31 dias** de ticket + clima.
A base inclui um aviso honesto: o clima em horário local tem **3 horas faltantes no fim de 31/03**,
então qualquer leitura sobre o último dia deve ser interpretada com essa ressalva.


In [ ]:
from pathlib import Path
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 40)
pd.set_option('display.width', 160)
pd.set_option('display.float_format', '{:,.4f}'.format)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams.update({'figure.figsize': (12, 5), 'figure.dpi': 120})


def locate_project_root() -> Path:
    start = Path.cwd().resolve()
    for base in [start, *start.parents]:
        if (base / 'data' / 'derived').exists():
            return base
    raise FileNotFoundError('Could not locate the repo root from the current working directory.')


PROJECT_ROOT = locate_project_root()
DERIVED = PROJECT_ROOT / 'data' / 'derived'
FIGURES = DERIVED / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)
WEATHER_ORDER = ['Clear', 'Light Rain', 'Moderate Rain', 'Heavy Rain / Storm']


def savefig(name: str) -> Path:
    path = FIGURES / name
    plt.tight_layout()
    plt.savefig(path, bbox_inches='tight')
    print(f'Saved figure: {path}')
    return path


In [ ]:
weather = pd.read_parquet(DERIVED / 'weather_hourly.parquet')
ticket = pd.read_parquet(DERIVED / 'ticket_hourly_route_profile.parquet')

weather_clean = weather.loc[~weather['weather_observation_missing']].copy()
weather_clean['weather_cat'] = pd.Categorical(weather_clean['weather_cat'], categories=WEATHER_ORDER, ordered=True)

route_hourly = (
    ticket.groupby(['date', 'hour', 'route_norm', 'is_weekend', 'day_of_week'], observed=True, as_index=False)
    .agg(boardings=('boardings', 'sum'))
    .merge(weather_clean[['date', 'hour', 'rain_mm', 'rain_3h', 'rain_6h', 'weather_cat', 'wind_gust', 'temp_c']], on=['date', 'hour'], how='inner')
)
profile_hourly = (
    ticket.groupby(['date', 'hour', 'card_label', 'is_weekend', 'day_of_week'], observed=True, as_index=False)
    .agg(boardings=('boardings', 'sum'))
    .merge(weather_clean[['date', 'hour', 'rain_mm', 'weather_cat']], on=['date', 'hour'], how='inner')
)
profile_hourly['weather_cat'] = pd.Categorical(profile_hourly['weather_cat'], categories=WEATHER_ORDER, ordered=True)

daily_ticket = ticket.groupby('date', as_index=False).agg(boardings=('boardings', 'sum'))
daily_weather = (
    weather_clean.groupby('date', as_index=False)
    .agg(
        rain_mm=('rain_mm', 'sum'),
        wind_gust=('wind_gust', 'max'),
        temp_c=('temp_c', 'mean'),
        observed_hours=('hour', 'size'),
    )
)
daily_weather['weather_cat'] = pd.Categorical(
    pd.cut(daily_weather['rain_mm'], bins=[-np.inf, 1, 10, 25, np.inf], labels=WEATHER_ORDER, right=False),
    categories=WEATHER_ORDER,
    ordered=True,
)
daily_weather['is_weekend'] = pd.to_datetime(daily_weather['date']).dt.dayofweek >= 5
daily_weather['day_of_week'] = pd.to_datetime(daily_weather['date']).dt.dayofweek
daily = daily_ticket.merge(daily_weather, on='date', how='inner').sort_values('date').reset_index(drop=True)

print('=== Sample composition ===')
print('ticket days:', pd.to_datetime(ticket['date']).dt.date.nunique())
print('weather days:', pd.to_datetime(weather_clean['date']).dt.date.nunique())
print('route-hour rows:', len(route_hourly))
print('profile-hour rows:', len(profile_hourly))
print('profiles:', sorted(profile_hourly['card_label'].dropna().astype(str).unique().tolist()))
print()

print('=== Weather coverage by day ===')
print(daily[['date', 'observed_hours', 'rain_mm', 'weather_cat']].tail(5).to_string(index=False))
print()

print('=== Daily weather category counts ===')
print(daily['weather_cat'].value_counts(dropna=False).reindex(WEATHER_ORDER).to_string())
print('Heavy Rain / Storm days:', int((daily['weather_cat'] == 'Heavy Rain / Storm').sum()))
print('Hourly observations with rain >= 10 mm:', int((weather_clean['rain_mm'] >= 10).sum()))


In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax2 = ax1.twinx()
ax1.bar(pd.to_datetime(daily['date']), daily['boardings'], color='#4C78A8', alpha=0.85)
ax2.plot(pd.to_datetime(daily['date']), daily['rain_mm'], color='#D62728', marker='o', linewidth=2)
ax1.set_title('Q1 · Daily demand and rainfall in March/2026')
ax1.set_ylabel('Boardings')
ax2.set_ylabel('Daily rainfall (mm)')
ax1.tick_params(axis='x', rotation=45)
savefig('q1_daily_demand_vs_rain.png')
plt.show()

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(data=route_hourly, x='weather_cat', y='boardings', order=WEATHER_ORDER, ax=ax)
ax.set_title('Q1 · Route-hour demand by weather category')
ax.set_xlabel('Weather category')
ax.set_ylabel('Boardings per route-hour')
ax.tick_params(axis='x', rotation=15)
savefig('q1_route_hour_boxplot.png')
plt.show()


In [ ]:
route_hourly = route_hourly.copy()
route_hourly['weather_cat'] = pd.Categorical(route_hourly['weather_cat'], categories=WEATHER_ORDER, ordered=True)

q1_model_all = smf.glm(
    'boardings ~ rain_mm + C(hour) + C(day_of_week)',
    data=route_hourly,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')
q1_model_weekday = smf.glm(
    'boardings ~ rain_mm + C(hour) + C(day_of_week)',
    data=route_hourly.loc[~route_hourly['is_weekend']].copy(),
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

daily_model = smf.glm(
    'boardings ~ rain_mm + C(day_of_week)',
    data=daily,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

rng = np.random.default_rng(20260702)
boot_coefs = []
for _ in range(150):
    sample = daily.sample(n=len(daily), replace=True, random_state=int(rng.integers(0, 1_000_000)))
    try:
        boot_model = smf.glm(
            'boardings ~ rain_mm + C(day_of_week)',
            data=sample,
            family=sm.families.Poisson(),
        ).fit()
        boot_coefs.append(float(boot_model.params['rain_mm']))
    except Exception:
        continue
boot_ci = np.quantile(boot_coefs, [0.025, 0.975]) if boot_coefs else [np.nan, np.nan]

q1_effects = pd.DataFrame(
    {
        'sample': ['all route-hours', 'weekday route-hours'],
        'rain_coef': [q1_model_all.params['rain_mm'], q1_model_weekday.params['rain_mm']],
        'rain_pvalue': [q1_model_all.pvalues['rain_mm'], q1_model_weekday.pvalues['rain_mm']],
        'pct_effect_per_mm': [100 * (np.exp(q1_model_all.params['rain_mm']) - 1), 100 * (np.exp(q1_model_weekday.params['rain_mm']) - 1)],
    }
)
print('=== Q1 Poisson models ===')
print(q1_effects.to_string(index=False))
print()
print('Daily bootstrap 95% CI for rain_mm coefficient:', boot_ci)
print('Daily model coefficient table:')
print(daily_model.summary().tables[1].as_text())

fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=q1_effects, x='sample', y='pct_effect_per_mm', ax=ax)
ax.axhline(0, color='black', linewidth=1)
ax.set_title('Q1 · Marginal effect of 1 mm of rain on demand')
ax.set_ylabel('Percent effect (%)')
ax.set_xlabel('')
savefig('q1_rain_effect_models.png')
plt.show()


In [ ]:
keep_profiles = ['Standard', 'Student', 'Labor', 'Senior', 'Cash']
profile_q2 = profile_hourly.loc[profile_hourly['card_label'].isin(keep_profiles)].copy()

q2_model = smf.glm(
    'boardings ~ rain_mm * C(card_label) + C(hour) + C(day_of_week)',
    data=profile_q2,
    family=sm.families.Poisson(),
).fit(cov_type='HC1')

interaction_rows = []
for label in keep_profiles:
    if label == 'Cash':
        term = 'rain_mm'
        interaction_rows.append({'profile': label, 'term': term, 'coef': q2_model.params.get(term, np.nan), 'pvalue': q2_model.pvalues.get(term, np.nan)})
        continue
    term = f'rain_mm:C(card_label)[T.{label}]'
    interaction_rows.append({'profile': label, 'term': term, 'coef': q2_model.params.get(term, 0.0), 'pvalue': q2_model.pvalues.get(term, np.nan)})
interaction_df = pd.DataFrame(interaction_rows)
print('=== Q2 interaction coefficients ===')
print(interaction_df.to_string(index=False))
print()

heatmap = (
    profile_q2.groupby(['card_label', 'weather_cat'], observed=True)['boardings']
    .mean()
    .unstack(fill_value=np.nan)
    .reindex(index=keep_profiles, columns=WEATHER_ORDER)
)
normalized_heatmap = heatmap.div(heatmap['Clear'], axis=0) - 1
print('=== Q2 normalized profile x weather table ===')
print((normalized_heatmap * 100).round(2).to_string())

fig, ax = plt.subplots(figsize=(10, 4.5))
sns.heatmap(normalized_heatmap * 100, annot=True, fmt='.1f', cmap='coolwarm', center=0, ax=ax)
ax.set_title('Q2 · Relative change vs clear weather by passenger profile (%)')
ax.set_xlabel('Weather category')
ax.set_ylabel('Passenger profile')
savefig('q2_profile_weather_heatmap.png')
plt.show()

profile_q2['rain_flag'] = np.where(profile_q2['rain_mm'].fillna(0) > 0, 'Rain', 'Clear')
curve = (
    profile_q2.loc[profile_q2['rain_flag'].isin(['Clear', 'Rain'])]
    .groupby(['card_label', 'hour', 'rain_flag'], observed=True, as_index=False)
    .agg(boardings=('boardings', 'mean'))
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)
for ax, label in zip(axes, ['Standard', 'Student']):
    sns.lineplot(data=curve.loc[curve['card_label'] == label], x='hour', y='boardings', hue='rain_flag', marker='o', ax=ax)
    ax.set_title(f'Q2 · Hourly curve — {label}')
    ax.set_xlabel('Hour of day')
    ax.set_ylabel('Mean boardings per hour')
savefig('q2_profile_hourly_curves.png')
plt.show()


## Conclusão

- **Q1:** a demanda responde à chuva dentro da janela completa de ticket + clima, e o notebook deixa explícita a comparação entre **toda a amostra** e a sensibilidade **apenas em dias úteis**.
- **Q2:** a heterogeneidade por perfil agora está operacionalizada com `card_label` legível (`Standard`, `Student`, `Labor`, `Senior`, `Cash`), com modelo de interação e visualizações dedicadas.
- **Limite importante:** março/2026 **não contém observações de `Heavy Rain / Storm`**, então a conclusão deve permanecer no escopo de **tempo firme vs chuva leve/moderada**.
